# Solace Spark Consumer Demo

This notebook demonstrates how to consume messages from Solace PubSub+ using the Spark Structured Streaming connector.

## Prerequisites

- Solace PubSub+ broker accessible
- Solace Spark Connector JAR installed on cluster (`com.solacecoe.connectors:pubsubplus-connector-spark:3.1.6`)
- A queue created on Solace with a topic subscription

## Configuration

In [ ]:
# Solace connection settings
SOLACE_HOST = "tcp://localhost:55554"  # Update for your environment
SOLACE_VPN = "default"
SOLACE_USERNAME = "default"
SOLACE_PASSWORD = "default"
SOLACE_QUEUE = "spark-consumer-queue"  # Must exist and have topic subscription

# Spark streaming settings
BATCH_SIZE = 100
PARTITIONS = 1  # Set to 0 for auto-scaling based on worker nodes

# Stream name for checkpointing
STREAM_NAME = "solace-spark-stream"

In [ ]:
import requests
from requests.auth import HTTPBasicAuth

# Solace SEMP API settings
SEMP_URL = "http://localhost:8081"  # Management port
VPN = "default"
QUEUE_NAME = "spark-consumer-queue"
TOPIC_SUBSCRIPTION = "synthetic/>"  # Matches your producer topic

# Create queue
queue_response = requests.post(
    f"{SEMP_URL}/SEMP/v2/config/msgVpns/{VPN}/queues",
    json={
        "queueName": QUEUE_NAME,
        "accessType": "exclusive",
        "egressEnabled": True,
        "ingressEnabled": True,
        "permission": "consume",
        "maxMsgSpoolUsage": 1500,
        "maxDeliveredUnackedMsgsPerFlow": 200,
    },
    auth=HTTPBasicAuth("admin", "admin"),
    headers={"Content-Type": "application/json"},
)
print(f"Queue creation: {queue_response.status_code}")

# Add topic subscription
sub_response = requests.post(
    f"{SEMP_URL}/SEMP/v2/config/msgVpns/{VPN}/queues/{QUEUE_NAME}/subscriptions",
    json={"subscriptionTopic": TOPIC_SUBSCRIPTION},
    auth=HTTPBasicAuth("admin", "admin"),
    headers={"Content-Type": "application/json"},
)
print(f"Subscription: {sub_response.status_code}")

## Create Streaming DataFrame from Solace

In [ ]:
df = spark.readStream \
    .format("solace") \
    .option("host", SOLACE_HOST) \
    .option("vpn", SOLACE_VPN) \
    .option("username", SOLACE_USERNAME) \
    .option("password", SOLACE_PASSWORD) \
    .option("queue", SOLACE_QUEUE) \
    .option("batchSize", BATCH_SIZE) \
    .option("partitions", PARTITIONS) \
    .option("includeHeaders", "true") \
    .option("connectRetries", 3) \
    .option("reconnectRetries", 3) \
    .load()

print("Streaming DataFrame schema:")
df.printSchema()

## Parse JSON Payload

In [ ]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

# Define schema for the synthetic event JSON
event_schema = StructType([
    StructField("event_id", StringType()),
    StructField("timestamp", StringType()),
    StructField("event_type", StringType()),
    StructField("user", StructType([
        StructField("user_id", StringType()),
        StructField("username", StringType()),
        StructField("email", StringType()),
        StructField("ip_address", StringType()),
    ])),
    StructField("device", StructType([
        StructField("type", StringType()),
        StructField("os", StringType()),
        StructField("browser", StringType()),
    ])),
    StructField("location", StructType([
        StructField("country", StringType()),
        StructField("city", StringType()),
        StructField("latitude", DoubleType()),
        StructField("longitude", DoubleType()),
    ])),
    StructField("metadata", StructType([
        StructField("session_id", StringType()),
        StructField("page_url", StringType()),
        StructField("referrer", StringType()),
        StructField("value", DoubleType()),
    ])),
])

# Parse the binary payload as JSON
df_parsed = df \
    .withColumn("payload_str", col("Payload").cast("string")) \
    .withColumn("event", from_json(col("payload_str"), event_schema)) \
    .select(
        col("Id").alias("message_id"),
        col("Topic"),
        col("TimeStamp").alias("receive_time"),
        col("event.*")
    )

## Option 1: Display Stream (Databricks)

Use `display()` to visualize the streaming data in Databricks:

In [ ]:
# Display streaming data (Databricks only)
# display(df_parsed)

## Option 2: Write to Delta Lake Table

In [ ]:
# Write to Delta Lake table
DELTA_TABLE = "solace_events"
CHECKPOINT_PATH = f"/tmp/checkpoints/{STREAM_NAME}"

query_delta = df_parsed.writeStream \
    .format("delta") \
    .outputMode("append") \
    .queryName(f"{STREAM_NAME}-delta") \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .toTable(DELTA_TABLE)

# Uncomment to start the stream:
# query_delta.awaitTermination()

## Option 3: Process with foreachBatch

Process each micro-batch with custom logic:

In [ ]:
def process_batch(batch_df, batch_id):
    """Process each micro-batch of messages."""
    count = batch_df.count()
    print(f"Batch {batch_id}: Processing {count} messages")
    
    if count > 0:
        # Example: Show event type distribution
        batch_df.groupBy("event_type").count().show()
        
        # Example: Write to table
        # batch_df.write.mode("append").saveAsTable("solace_events")

CHECKPOINT_PATH = f"/tmp/checkpoints/{STREAM_NAME}-foreach"

query_foreach = df_parsed.writeStream \
    .foreachBatch(process_batch) \
    .outputMode("append") \
    .queryName(f"{STREAM_NAME}-foreach") \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .start()

# Uncomment to start the stream:
# query_foreach.awaitTermination()

## Query Consumed Data

Once data is written to Delta Lake, you can query it:

In [ ]:
# Read from Delta table
# events_df = spark.table("solace_events")

# Event count by type
# events_df.groupBy("event_type").count().orderBy("count", ascending=False).show()

# Recent events
# events_df.orderBy(col("timestamp").desc()).limit(10).show(truncate=False)

# Events by country
# events_df.groupBy("location.country").count().orderBy("count", ascending=False).show()

## Stop Streaming Queries

In [ ]:
# List active streaming queries
for q in spark.streams.active:
    print(f"Query: {q.name}, Status: {q.status}")

# Stop all streaming queries
# for q in spark.streams.active:
#     q.stop()

## Reference

### Solace Source Schema

The Solace Spark Connector provides these columns:

| Column | Type | Description |
|--------|------|-------------|
| Id | String | Message ID (replication group message ID by default) |
| Payload | Binary | Message payload |
| PartitionKey | String | Partition key if present |
| Topic | String | Topic the message was published to |
| TimeStamp | Timestamp | Sender timestamp or receive time |
| Headers | Map<String, Binary> | Message headers (if includeHeaders=true) |

### Key Connector Options

| Option | Default | Description |
|--------|---------|-------------|
| host | - | Solace broker URL (tcp://host:port) |
| vpn | - | Message VPN name |
| username | - | Client username |
| password | - | Client password |
| queue | - | Queue name to consume from |
| batchSize | 1 | Messages per micro-batch |
| partitions | 1 | Number of consumers (0=auto-scale) |
| includeHeaders | false | Include message headers |
| connectRetries | 0 | Connection retry attempts |
| reconnectRetries | 3 | Reconnection retry attempts |